<a href="https://colab.research.google.com/github/MrDev333/PhishingDetect/blob/backend/notebooks/ResolveURLmk3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd

df = pd.read_csv('/content/online-valid_1.csv')
display(df.head())

,phish_id,url,phish_detail_url,submission_time,verified,verification_time,online,target
0,9190066,https://allegrolokalnie.pl-oferta6542763.sbs/?...,http://www.phishtank.com/phish_detail.php?phis...,2025-08-21T00:06:16+00:00,yes,2025-08-21T00:12:10+00:00,yes,Allegro
1,9190064,https://yellowcardapi.webflow.io/,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T23:55:26+00:00,yes,2025-08-21T00:03:36+00:00,yes,Other
2,9190063,http://yellowcardapi.webflow.io,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T23:55:25+00:00,yes,2025-08-21T00:03:36+00:00,yes,Other
3,9190061,http://allegro.pl-oferta5353452.icu,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T23:44:25+00:00,yes,2025-08-20T23:51:41+00:00,yes,Allegro
4,9190060,https://cancelamentodigitalbia.click/?a5125=ds...,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T23:42:23+00:00,yes,2025-08-20T23:51:41+00:00,yes,Other


In [3]:
# Data Preprocessing: Extract features from URLs
from urllib.parse import urlparse
import re

def extract_features(url):
    features = {}
    try:
        parsed_url = urlparse(url)
        features['scheme'] = parsed_url.scheme
        features['netloc'] = parsed_url.netloc
        features['path'] = parsed_url.path
        features['params'] = parsed_url.params
        features['query'] = parsed_url.query
        features['fragment'] = parsed_url.fragment
        features['domain'] = parsed_url.hostname
        features['length'] = len(url)
        features['num_dots'] = url.count('.')
        features['num_hyphens'] = url.count('-')
        features['num_at'] = url.count('@')
        features['num_question'] = url.count('?')
        features['num_ampersand'] = url.count('&')
        features['num_equals'] = url.count('=')
        features['num_exclamation'] = url.count('!')
        features['num_space'] = url.count(' ')
        features['num_tilde'] = url.count('~')
        features['num_comma'] = url.count(',')
        features['num_plus'] = url.count('+')
        features['num_asterisk'] = url.count('*')
        features['num_hash'] = url.count('#')
        features['num_dollar'] = url.count('$')
        features['num_percent'] = url.count('%')
        features['has_http'] = 'http' in parsed_url.scheme
        features['has_https'] = 'https' in parsed_url.scheme
        features['has_ftp'] = 'ftp' in parsed_url.scheme
        features['has_email'] = '@' in url

        # Check for IP address in domain
        if features['domain']:
            features['is_ip'] = bool(re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', features['domain']))
        else:
            features['is_ip'] = False


    except:
        pass # Handle potential errors during parsing
    return features

df_features = df['url'].apply(lambda x: pd.Series(extract_features(x)))

# Combine original dataframe with features
df_processed = pd.concat([df, df_features], axis=1)

display(df_processed.head())

,phish_id,url,phish_detail_url,submission_time,verified,verification_time,online,target,scheme,netloc,...,num_plus,num_asterisk,num_hash,num_dollar,num_percent,has_http,has_https,has_ftp,has_email,is_ip
0,9190066,https://allegrolokalnie.pl-oferta6542763.sbs/?...,http://www.phishtank.com/phish_detail.php?phis...,2025-08-21T00:06:16+00:00,yes,2025-08-21T00:12:10+00:00,yes,Allegro,https,allegrolokalnie.pl-oferta6542763.sbs,...,0,0,0,0,0,True,True,False,False,False
1,9190064,https://yellowcardapi.webflow.io/,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T23:55:26+00:00,yes,2025-08-21T00:03:36+00:00,yes,Other,https,yellowcardapi.webflow.io,...,0,0,0,0,0,True,True,False,False,False
2,9190063,http://yellowcardapi.webflow.io,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T23:55:25+00:00,yes,2025-08-21T00:03:36+00:00,yes,Other,http,yellowcardapi.webflow.io,...,0,0,0,0,0,True,False,False,False,False
3,9190061,http://allegro.pl-oferta5353452.icu,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T23:44:25+00:00,yes,2025-08-20T23:51:41+00:00,yes,Allegro,http,allegro.pl-oferta5353452.icu,...,0,0,0,0,0,True,False,False,False,False
4,9190060,https://cancelamentodigitalbia.click/?a5125=ds...,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T23:42:23+00:00,yes,2025-08-20T23:51:41+00:00,yes,Other,https,cancelamentodigitalbia.click,...,0,0,0,0,0,True,True,False,False,False


# Task
Analyze the data in the file "/content/online-valid_1.csv" to identify suspicious URLs.

## Feature selection

### Subtask:
Choose the most relevant features from the extracted URL features for identifying suspicious links.


**Reasoning**:
I will examine the `df_processed` DataFrame to identify the extracted URL features and select the most relevant ones for identifying suspicious links based on common phishing characteristics.



In [4]:
# Identify the columns that represent the extracted URL features
original_columns = df.columns.tolist()
all_columns = df_processed.columns.tolist()
extracted_features_columns = [col for col in all_columns if col not in original_columns]

# Select relevant features based on common phishing URL characteristics
selected_features = [
    'length',          # Longer URLs can be suspicious
    'num_dots',        # Excessive dots can indicate subdomains or obfuscation
    'num_hyphens',     # Hyphens can be used to mimic legitimate domains
    'num_at',          # The '@' symbol can be used to embed credentials
    'num_question',    # Presence of query parameters
    'num_ampersand',   # Multiple parameters in query string
    'num_equals',      # Multiple assignments in query string
    'has_http',        # Check for http (less secure)
    'has_https',       # Check for https (more secure, but phishers use it too)
    'is_ip',           # Using an IP address instead of a domain name
    'netloc',          # The network location part of the URL
    'path',            # The path part of the URL
    'query'            # The query part of the URL

]

# You can further refine this list based on domain knowledge or feature importance analysis later.
# For this subtask, this selection is based on general understanding of phishing URLs.

print("Selected features for identifying suspicious URLs:")
print(selected_features)

Selected features for identifying suspicious URLs:
['length', 'num_dots', 'num_hyphens', 'num_at', 'num_question', 'num_ampersand', 'num_equals', 'has_http', 'has_https', 'is_ip', 'netloc', 'path', 'query']


## Model training/rule definition

### Subtask:
Define rules based on the selected features to identify potentially suspicious URLs.


**Reasoning**:
Initialize the 'is_suspicious' column and define and apply rules based on selected features to identify potentially suspicious URLs.



In [5]:
# Create a new column and initialize to False
df_processed['is_suspicious'] = False

# Define rules based on selected features
# Rule 1: Excessive number of dots in netloc (heuristic: more than 4 dots)
rule_excessive_dots = df_processed['netloc'].str.count(r'\.') > 4

# Rule 2: Excessive number of hyphens in netloc (heuristic: more than 5 hyphens)
rule_excessive_hyphens = df_processed['netloc'].str.count('-') > 5

# Rule 3: Presence of "@" symbol in the URL (can be used to hide the true domain)
rule_at_symbol = df_processed['num_at'] > 0

# Rule 4: Using an IP address instead of a domain name
rule_is_ip = df_processed['is_ip'] == True

# Rule 5: Very long URLs (heuristic: length greater than 100)
rule_long_url = df_processed['length'] > 100

# Rule 6: Very long path (heuristic: path length greater than 50)
rule_long_path = df_processed['path'].str.len() > 50

# Rule 7: Very long query string (heuristic: query length greater than 50)
rule_long_query = df_processed['query'].str.len() > 50


# Combine rules using logical OR
suspicious_condition = (
    rule_excessive_dots |
    rule_excessive_hyphens |
    rule_at_symbol |
    rule_is_ip |
    rule_long_url |
    rule_long_path |
    rule_long_query
)

# Apply the rules to update the 'is_suspicious' column
df_processed.loc[suspicious_condition, 'is_suspicious'] = True

# Display the count of suspicious URLs
display(df_processed['is_suspicious'].value_counts())

,count
is_suspicious,
False,43205
True,9689


## Analysis of suspicious links

### Subtask:
Examine the identified suspicious links and their associated features to understand the patterns of malicious URLs in this dataset.


**Reasoning**:
Filter the dataframe to get suspicious URLs, display the head, analyze the target column, and calculate descriptive statistics for numerical features used in the rules.



In [6]:
# 1. Filter df_processed to get a DataFrame containing only the rows where is_suspicious is True.
suspicious_urls_df = df_processed[df_processed['is_suspicious'] == True].copy()

# 2. Display the first 5 rows of suspicious_urls_df
print("First 5 rows of suspicious URLs DataFrame:")
display(suspicious_urls_df.head())

# 3. Analyze the value counts of the 'target' column for the suspicious_urls_df
print("\nValue counts of 'target' for suspicious URLs:")
display(suspicious_urls_df['target'].value_counts().head()) # Displaying head for brevity

# 4. Calculate and display descriptive statistics for numerical features used in the rules
numerical_features_for_stats = ['length', 'num_dots', 'num_hyphens', 'num_at', 'num_question', 'num_ampersand', 'num_equals']
print("\nDescriptive statistics for numerical features of suspicious URLs:")
display(suspicious_urls_df[numerical_features_for_stats].describe())

First 5 rows of suspicious URLs DataFrame:


,phish_id,url,phish_detail_url,submission_time,verified,verification_time,online,target,scheme,netloc,...,num_asterisk,num_hash,num_dollar,num_percent,has_http,has_https,has_ftp,has_email,is_ip,is_suspicious
38,9189981,https://sso--i-trustcapital---com-sso.webflow.io/,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T20:27:10+00:00,yes,2025-08-20T20:41:55+00:00,yes,Other,https,sso--i-trustcapital---com-sso.webflow.io,...,0,0,0,0,True,True,False,False,False,True
40,9189979,https://sso--itrust-capital-com-auth-sso.webfl...,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T20:27:04+00:00,yes,2025-08-20T20:41:55+00:00,yes,Other,https,sso--itrust-capital-com-auth-sso.webflow.io,...,0,0,0,0,True,True,False,False,False,True
41,9189978,https://doc--ledgre--en-us-apps.webflow.io/,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T20:27:01+00:00,yes,2025-08-20T20:41:55+00:00,yes,Other,https,doc--ledgre--en-us-apps.webflow.io,...,0,0,0,0,True,True,False,False,False,True
46,9189973,https://auth--apap--bitmert--sso.webflow.io/,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T20:26:45+00:00,yes,2025-08-20T20:41:55+00:00,yes,Other,https,auth--apap--bitmert--sso.webflow.io,...,0,0,0,0,True,True,False,False,False,True
64,9189955,https://sso--i----trustcapital-com-autths.webf...,http://www.phishtank.com/phish_detail.php?phis...,2025-08-20T20:25:27+00:00,yes,2025-08-20T20:41:55+00:00,yes,Other,https,sso--i----trustcapital-com-autths.webflow.io,...,0,0,0,0,True,True,False,False,False,True



Value counts of 'target' for suspicious URLs:


,count
target,
Other,9098
Internal Revenue Service,351
Optus,39
Microsoft,35
Allegro,24



Descriptive statistics for numerical features of suspicious URLs:


,length,num_dots,num_hyphens,num_at,num_question,num_ampersand,num_equals
count,9689.000000,9689.000000,9689.000000,9689.000000,9689.000000,9689.000000,9689.000000
mean,172.619053,2.859944,2.037052,0.047064,0.852823,1.933430,2.783053
std,378.946855,2.011552,2.213653,0.228208,0.540488,2.451344,2.844003
min,20.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,119.000000,2.000000,1.000000,0.000000,1.000000,0.000000,1.000000
50%,175.000000,2.000000,2.000000,0.000000,1.000000,2.000000,3.000000
75%,175.000000,3.000000,3.000000,0.000000,1.000000,2.000000,3.000000
max,25523.000000,23.000000,50.000000,8.000000,17.000000,43.000000,47.000000


**Reasoning**:
Examine the distribution of boolean features for the suspicious URLs and summarize the observations about the characteristics of suspicious URLs based on the analysis.



In [7]:
# 5. Examine the distribution of boolean features ('has_http', 'has_https', 'is_ip') for the suspicious_urls_df
print("\nValue counts of boolean features for suspicious URLs:")
display(suspicious_urls_df[['has_http', 'has_https', 'is_ip']].apply(pd.value_counts))

# 6. Summarize observations
print("\nSummary of Observations:")
print("- The majority of suspicious URLs target 'Other' categories, followed by specific entities like 'Internal Revenue Service', 'Optus', 'Microsoft', and 'Allegro'.")
print("- Suspicious URLs tend to be longer on average (mean length ~172) compared to typical URLs.")
print("- Suspicious URLs often contain multiple dots, hyphens, question marks, ampersands, and equal signs, indicating complex subdomains, obfuscation, and extensive query parameters.")
print("- While most suspicious URLs use HTTPS, a significant portion still use HTTP.")
print("- A small percentage of suspicious URLs use IP addresses instead of domain names.")
print("- The presence of '@' symbol, excessive dots, excessive hyphens, long URLs, long paths, and long queries are the key indicators based on the defined rules.")


Value counts of boolean features for suspicious URLs:


/tmp/ipython-input-4079856345.py:3: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  display(suspicious_urls_df[['has_http', 'has_https', 'is_ip']].apply(pd.value_counts))


,has_http,has_https,is_ip
False,NaN,561,9340
True,9689.0,9128,349



Summary of Observations:
- The majority of suspicious URLs target 'Other' categories, followed by specific entities like 'Internal Revenue Service', 'Optus', 'Microsoft', and 'Allegro'.
- Suspicious URLs tend to be longer on average (mean length ~172) compared to typical URLs.
- Suspicious URLs often contain multiple dots, hyphens, question marks, ampersands, and equal signs, indicating complex subdomains, obfuscation, and extensive query parameters.
- While most suspicious URLs use HTTPS, a significant portion still use HTTP.
- A small percentage of suspicious URLs use IP addresses instead of domain names.
- The presence of '@' symbol, excessive dots, excessive hyphens, long URLs, long paths, and long queries are the key indicators based on the defined rules.


## Summary:

### Data Analysis Key Findings

*   9,689 URLs were identified as potentially suspicious based on the defined rules, while 43,205 were not.
*   Suspicious URLs tend to be longer, with a mean length of approximately 172 characters.
*   Suspicious URLs frequently contain multiple dots, hyphens, question marks, ampersands, and equal signs, indicating complex structures and extensive query parameters.
*   The majority of suspicious URLs target "Other" categories, followed by specific entities like the Internal Revenue Service, Optus, Microsoft, and Allegro.
*   While most suspicious URLs use HTTPS, a significant number still use HTTP.
*   A small percentage of suspicious URLs use IP addresses instead of domain names.

### Insights or Next Steps

*   The current rules identify a substantial number of potentially suspicious URLs. Further refinement of these rules or exploration of machine learning models could improve accuracy and identify more nuanced patterns.
*   Investigating the "Other" category of targets for suspicious URLs could reveal new or emerging phishing trends.


In [11]:
%pip install tqdm

In [19]:
# Prepare data for classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Select features and target variable
# Exclude non-numeric and potentially problematic columns
features = df_processed.drop(['url', 'scheme', 'netloc', 'path', 'params', 'query', 'fragment', 'domain', 'phish_detail_url', 'submission_time', 'verification_time', 'target', 'phish_id', 'online'], axis=1)
target = df_processed['is_suspicious']

# Convert boolean and object type columns to numeric (0 or 1)
for col in features.columns:
    if features[col].dtype == 'bool':
        features[col] = features[col].astype(int)
    elif features[col].dtype == 'object':
        # Assuming 'yes' and 'no' are the only string values in object columns
        features[col] = features[col].map({'yes': 1, 'no': 0})


# Handle potential missing values by filling with 0 (or another appropriate strategy)
features = features.fillna(0)


# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)

# Train a classification model (e.g., RandomForestClassifier)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Accuracy Score:", accuracy_score(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

       False       1.00      1.00      1.00      8690
        True       1.00      1.00      1.00      1889

    accuracy                           1.00     10579
   macro avg       1.00      1.00      1.00     10579
weighted avg       1.00      1.00      1.00     10579

Accuracy Score: 1.0


In [20]:
# Filter the DataFrame to get only the suspicious URLs
suspicious_urls_df = df_processed[df_processed['is_suspicious'] == True]

# Select the 'url' column
blacklist_urls = suspicious_urls_df['url']

# Save the suspicious URLs to a new CSV file
blacklist_urls.to_csv('blacklist.csv', index=False, header=False)

print("Suspicious URLs saved to blacklist.csv")

Suspicious URLs saved to blacklist.csv


In [25]:
# Filter the DataFrame to get only the non-suspicious URLs (whitelist)
non_suspicious_urls_df = df_processed[df_processed['is_suspicious'] == False]

# Select the 'url' column
whitelist_urls = non_suspicious_urls_df['url']

# Save the non-suspicious URLs to a new CSV file
whitelist_urls.to_csv('whitelist.csv', index=False, header=False)

print("Non-suspicious URLs saved to whitelist.csv")

Non-suspicious URLs saved to whitelist.csv
